# 03 — Option Surface Distributions

**Purpose:** characterize the 0DTE option-surface features before any of them enter the regime feature contract, and test the three surface questions raised by the project audit:

| Audit item | Question tested here |
|---|---|
| **A2** | Does the fixed log-moneyness grid ($k = \pm2.5\%, \pm5\%$) mean the same thing at different times to expiry and across assets? Would a standardized-moneyness grid $z = k / (\sigma_{ATM}\sqrt{\tau})$ be more stationary? |
| **B2** | How much of the "preferred" surface is Deribit's mark model rather than market quotes, and do switches between the two sources create artificial jumps in surface changes? |
| **B1** | How many valid option snapshots are lost because surface eligibility requires the futures stream? |
| **B4** | Do features behave differently inside the 07:30–08:00 UTC settlement window? |

This notebook:

- opens `research.duckdb` **read-only**, in a **UTC** session;
- uses `time_series_feature_state` for the model-facing features and the long-form option views (`normalized_option_chain`, `smile_node_state`) for diagnostics the feature table cannot answer;
- builds a **standardized-moneyness prototype in memory only**. Nothing is written to the database.

It does **not**:

- impute, winsorize, or standardize anything;
- fit an SSVI or any arbitrage-free surface;
- change the production surface (if the evidence favours $z$-space, the change goes into `src/data_engineering/surfaces/` as a separate, reviewed step);
- define final time-to-expiry bins. The buckets below are exploratory.

> Time to expiry and time of day are the same clock for a daily 0DTE (expiry 08:00 UTC). Every "TTE effect" in this notebook is also an intraday-seasonality effect.

## 1. Environment and read-only connection

In [ ]:
from pathlib import Path
import os
import math
import time
import warnings

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 220)
pd.set_option("display.width", 200)

load_dotenv()

data_root_env = os.getenv("DERIBIT_DATA_ROOT")
if not data_root_env:
    raise RuntimeError("DERIBIT_DATA_ROOT is not defined in the environment/.env file.")

DATA_ROOT = Path(data_root_env).expanduser().resolve()
DB_PATH = DATA_ROOT / "database" / "research.duckdb"
if not DB_PATH.exists():
    raise FileNotFoundError(f"Research database not found: {DB_PATH}")

con = duckdb.connect(str(DB_PATH), read_only=True)
con.execute("SET TimeZone = 'UTC'")

print(f"Research DB:        {DB_PATH}")
print("Connection mode:    read-only")
print("Session time zone:", con.execute("SELECT current_setting('TimeZone')").fetchone()[0])


def q(sql, label):
    """Run a read-only query and report its size and runtime."""
    started = time.perf_counter()
    df = con.execute(sql).df()
    print(f"[{label}] {len(df):,} rows in {time.perf_counter() - started:,.1f}s")
    return df


def relation_columns(name):
    return set(con.execute(f"DESCRIBE SELECT * FROM {name}").df()["column_name"])

## 2. Load the model-facing option features

Feature families:

- **Fixed-$k$ levels:** ATM IV, symmetric IV differences ("skew") and curvature at $k=\pm2.5\%$ and $\pm5\%$, local skew slopes.
- **Delta levels:** 25Δ/10Δ put and call IV, risk reversals (RR) and butterflies (BF).
- **Dynamics:** 5m / 15m / 1h changes, blocked across the 08:00 session boundary by the pipeline.
- **Quality metadata:** quality tier, usability flags, interpolation spans, eligible rows and option spreads.

IV units are detected from the data and printed; the standardized-moneyness calculation converts to decimals.

In [ ]:
features = con.execute(
    "SELECT * FROM time_series_feature_state ORDER BY state_time, underlying"
).df()
features["state_time"] = pd.to_datetime(features["state_time"], utc=True)

OPTION_LEVEL = [
    "option_iv_atm_forward",
    "option_skew_025", "option_skew_050",
    "option_curvature_025", "option_curvature_050",
    "option_local_skew_slope_025", "option_local_skew_slope_050",
    "iv_put_25d", "iv_call_25d", "rr_25d", "bf_25d",
    "iv_put_10d", "iv_call_10d", "rr_10d", "bf_10d",
]
OPTION_DYNAMIC = [
    "atm_iv_change_5m", "atm_iv_change_15m", "atm_iv_change_1h",
    "skew_025_change_5m", "skew_025_change_15m", "skew_025_change_1h",
    "curvature_025_change_5m", "curvature_025_change_15m", "curvature_025_change_1h",
    "skew_050_change_5m", "rr_25d_change_5m", "bf_25d_change_5m",
]
OPTION_QUALITY = [
    "core_smile_quality_tier", "core_smile_tight", "core_smile_usable",
    "wing_050_usable", "full_grid_usable",
    "core_max_interpolation_span", "wing_050_max_interpolation_span",
    "full_grid_max_interpolation_span",
    "delta_25d_smile_available", "full_delta_smile_available",
    "max_delta_interpolation_span", "median_delta_interpolation_span",
    "option_base_eligible_rows", "option_two_sided_eligible_rows",
    "option_median_two_sided_spread_bps", "option_p90_two_sided_spread_bps",
]

for name, cols in [("level", OPTION_LEVEL), ("dynamic", OPTION_DYNAMIC), ("quality", OPTION_QUALITY)]:
    missing = [c for c in cols if c not in features.columns]
    print(f"{name:<8} {len(cols) - len(missing):>2} / {len(cols)} available" + (f" | missing: {missing}" if missing else ""))

OPTION_LEVEL = [c for c in OPTION_LEVEL if c in features.columns]
OPTION_DYNAMIC = [c for c in OPTION_DYNAMIC if c in features.columns]

# IV units: Deribit publishes IV in percent. Detect rather than assume.
atm_median = pd.to_numeric(features["option_iv_atm_forward"], errors="coerce").median()
IV_TO_DECIMAL = 0.01 if atm_median > 3 else 1.0
print(f"\nMedian ATM IV: {atm_median:.4g} -> IV unit = {'percent' if IV_TO_DECIMAL == 0.01 else 'decimal'}")

MINUTES_PER_YEAR = 365.0 * 24.0 * 60.0
features["tte_years"] = features["minutes_to_expiry"] / MINUTES_PER_YEAR
features["atm_sigma_sqrt_t"] = (
    pd.to_numeric(features["option_iv_atm_forward"], errors="coerce") * IV_TO_DECIMAL
    * np.sqrt(features["tte_years"])
)

# Exploratory buckets only; they are not model boundaries.
TTE_EDGES = [0, 30, 60, 180, 360, 720, 1080, 1440]
TTE_LABELS = ["≤30m (settle)", "30m–1h", "1–3h", "3–6h", "6–12h", "12–18h", "18–24h"]
features["tte_bucket"] = pd.cut(features["minutes_to_expiry"], bins=TTE_EDGES, labels=TTE_LABELS, right=True)
features["in_settlement_window"] = features["minutes_to_expiry"] <= 30

ASSETS = sorted(features["underlying"].dropna().unique())
print(f"Rows: {len(features):,} | assets: {ASSETS}")
print(f"States: {features['state_time'].min()} → {features['state_time'].max()}")

In [ ]:
CENTRAL_LOWER_Q, CENTRAL_UPPER_Q = 0.005, 0.995
TAIL_QS = [0.001, 0.01, 0.05, 0.50, 0.95, 0.99, 0.999]


def clean_numeric(df, column):
    return pd.to_numeric(df[column], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()


def feature_summary(df, columns, group_col=None):
    rows = []
    iterator = [("ALL", df)] if group_col is None else df.groupby(group_col, dropna=False, observed=True)
    for group_name, group in iterator:
        for col in columns:
            if col not in group.columns:
                continue
            s = clean_numeric(group, col)
            if s.empty:
                continue
            qv = s.quantile(TAIL_QS)
            rows.append({
                "group": group_name, "feature": col, "n": len(s),
                "missing_pct": 100.0 * (len(group) - len(s)) / max(len(group), 1),
                "mean": s.mean(), "std": s.std(), "skew": s.skew(), "excess_kurtosis": s.kurt(),
                "min": s.min(), "q001": qv.loc[0.001], "q01": qv.loc[0.01], "q05": qv.loc[0.05],
                "median": qv.loc[0.50], "q95": qv.loc[0.95], "q99": qv.loc[0.99], "q999": qv.loc[0.999],
                "max": s.max(),
            })
    return pd.DataFrame(rows)


def plot_asset_histograms(df, column, bins=60):
    ncols = 2
    nrows = math.ceil(len(ASSETS) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.4 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, asset in zip(axes, ASSETS):
        s = clean_numeric(df[df["underlying"] == asset], column)
        if s.empty:
            ax.text(0.5, 0.5, "No observations", ha="center", va="center")
        else:
            lo, hi = s.quantile(CENTRAL_LOWER_Q), s.quantile(CENTRAL_UPPER_Q)
            ax.hist(s[(s >= lo) & (s <= hi)], bins=bins)
        ax.set_title(str(asset))
    for ax in axes[len(ASSETS):]:
        ax.axis("off")
    fig.suptitle(f"{column} — per-asset central 99%", y=1.01)
    plt.tight_layout()
    plt.show()


def as_bool(s):
    """Nullable/boolean-like column -> plain bool (missing = False)."""
    if s.dtype == bool:
        return s
    mapped = s.map(lambda v: str(v).strip().lower() in ("true", "1", "1.0"))
    return mapped.astype(bool)


def median_by_tte(df, column, assets=None):
    """Median of a feature at each minutes_to_expiry, per asset."""
    assets = assets or ASSETS
    out = (
        df[df["underlying"].isin(assets)]
        .assign(_v=lambda d: pd.to_numeric(d[column], errors="coerce"))
        .groupby(["underlying", "minutes_to_expiry"])["_v"]
        .median()
        .unstack("underlying")
        .sort_index(ascending=False)
    )
    return out


def plot_median_by_tte(df, columns, title, logy=False):
    fig, axes = plt.subplots(1, len(columns), figsize=(6.5 * len(columns), 4.6), squeeze=False)
    for ax, col in zip(axes[0], columns):
        m = median_by_tte(df, col)
        for asset in m.columns:
            ax.plot(m.index / 60.0, m[asset], linewidth=1.2, label=str(asset))
        ax.invert_xaxis()
        ax.set_xlabel("hours to expiry")
        ax.set_title(col)
        if logy:
            ax.set_yscale("log")
    axes[0][0].legend(fontsize=8)
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


def eta_squared(df, column, group_col="tte_bucket"):
    """Share of a feature's variance explained by bucket medians (within one asset).

    0 means the clock explains nothing; 1 means the feature is a deterministic
    function of the clock bucket.
    """
    d = df[[column, group_col]].copy()
    d[column] = pd.to_numeric(d[column], errors="coerce")
    d = d.dropna()
    if len(d) < 50 or d[column].var() == 0:
        return np.nan
    group_mean = d.groupby(group_col, observed=True)[column].transform("mean")
    return float(((group_mean - d[column].mean()) ** 2).sum() / ((d[column] - d[column].mean()) ** 2).sum())


def within_session_acf(df, column, max_lag=24):
    """ACF on the exact 5-minute grid, never crossing a session boundary."""
    rows = []
    for asset, g in df.groupby("underlying"):
        g = g.sort_values("state_time")
        x = pd.to_numeric(g[column], errors="coerce")
        for lag in range(1, max_lag + 1):
            lx = x.shift(lag)
            ok = (
                x.notna() & lx.notna()
                & ((g["state_time"] - g["state_time"].shift(lag)) == pd.Timedelta(minutes=5 * lag))
                & (g["session_id"] == g["session_id"].shift(lag))
            )
            rows.append({
                "underlying": asset, "feature": column, "lag_minutes": 5 * lag,
                "n_pairs": int(ok.sum()),
                "acf": x[ok].corr(lx[ok]) if ok.sum() >= 20 else np.nan,
            })
    return pd.DataFrame(rows)

# Part A — Coverage

## 3. Option snapshots lost to the futures gate (audit B1)

Surface eligibility in `option_normalization.py` requires `state_research_ready`, which requires a complete **futures** stream. The option forward is built from the option chain itself, so the surface does not need the futures.

`lost_to_futures_gate` counts states with a valid option snapshot (`option_snapshot_ready`) but no surface because the futures side was not ready. These states include every session before the futures collector started and every futures reconnect.

In [ ]:
cms_cols = relation_columns("canonical_market_state")
needed = {"option_snapshot_ready", "state_research_ready", "price_features_ready"}
if needed.issubset(cms_cols):
    gate = q(
        """
        SELECT
            underlying,
            expiry_date,
            COUNT(*) AS states,
            SUM(CASE WHEN option_snapshot_ready THEN 1 ELSE 0 END) AS option_ready,
            SUM(CASE WHEN state_research_ready THEN 1 ELSE 0 END) AS research_ready,
            SUM(CASE WHEN option_snapshot_ready AND NOT COALESCE(state_research_ready, FALSE)
                     THEN 1 ELSE 0 END) AS lost_to_futures_gate,
            SUM(CASE WHEN option_snapshot_ready AND NOT COALESCE(price_features_ready, FALSE)
                     THEN 1 ELSE 0 END) AS option_ok_futures_not_ready
        FROM canonical_market_state
        GROUP BY 1, 2
        ORDER BY 2, 1
        """,
        "futures-gate loss",
    )
    by_asset = gate.groupby("underlying")[["states", "option_ready", "research_ready", "lost_to_futures_gate"]].sum()
    by_asset["lost_pct_of_option_ready"] = 100 * by_asset["lost_to_futures_gate"] / by_asset["option_ready"]
    display(by_asset.round(2))
    display(
        gate.pivot_table(index="expiry_date", columns="underlying", values="lost_to_futures_gate", aggfunc="sum")
    )
else:
    print(f"canonical_market_state lacks {needed - cms_cols}; section skipped.")
    gate = pd.DataFrame()

## 4. Surface availability through the session

Which surface representations exist at each time to expiry? The fixed-$k$ core needs strikes bracketing $k=0, \pm2.5\%$; the delta smile needs options with $|\Delta|$ around 0.25 and 0.10, which disappear near expiry as deltas collapse to 0 or 1.

In [ ]:
avail_cols = [c for c in [
    "core_smile_usable", "core_smile_tight", "wing_050_usable", "full_grid_usable",
    "delta_25d_smile_available", "full_delta_smile_available",
] if c in features.columns]

opt_rows = features[features["primary_option_family_id"].notna()] if "primary_option_family_id" in features.columns else features
availability = (
    opt_rows.assign(**{c: as_bool(opt_rows[c]) for c in avail_cols})
    .groupby(["underlying", "tte_bucket"], observed=True)[avail_cols]
    .mean()
    .mul(100)
    .round(1)
)
print("Share of states with a primary option family where each representation is available (%):")
display(availability)

if "core_smile_quality_tier" in features.columns:
    tiers = (
        opt_rows.groupby(["underlying", "core_smile_quality_tier"]).size()
        .unstack(fill_value=0)
    )
    display(tiers.div(tiers.sum(axis=1), axis=0).mul(100).round(1))

fig, axes = plt.subplots(1, len(avail_cols), figsize=(4.2 * len(avail_cols), 3.8), sharey=True)
for ax, col in zip(np.atleast_1d(axes), avail_cols):
    m = (
        opt_rows.assign(_v=as_bool(opt_rows[col]).astype(float))
        .groupby(["minutes_to_expiry", "underlying"])["_v"].mean().unstack()
        .sort_index(ascending=False)
    )
    for asset in m.columns:
        ax.plot(m.index / 60, 100 * m[asset], linewidth=1, label=asset)
    ax.invert_xaxis()
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("hours to expiry")
np.atleast_1d(axes)[0].set_ylabel("% of states")
np.atleast_1d(axes)[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

# Part B — Level Distributions

## 5. Pooled and per-asset distributions

ATM IV levels differ several-fold across assets, so pooled quantiles mainly describe the asset mix. The per-asset table is the relevant one.

In [ ]:
level_summary = feature_summary(features, OPTION_LEVEL)
level_by_asset = feature_summary(features, OPTION_LEVEL, group_col="underlying")
display(level_summary)
display(level_by_asset)

for col in ["option_iv_atm_forward", "option_skew_025", "option_curvature_025", "rr_25d", "bf_25d"]:
    if col in features.columns:
        plot_asset_histograms(features, col)

# Part C — Does Fixed Log-Moneyness Mean the Same Thing All Session? (audit A2)

## 6. The ±2.5% node in units of expected move

A 0DTE option's one-standard-deviation move to expiry is $\sigma_{ATM}\sqrt{\tau}$, which shrinks through the session. The fixed node $k = \pm 2.5\%$ sits at

$$
z_{2.5\%} = \frac{0.025}{\sigma_{ATM}\,\sqrt{\tau}}
$$

standard deviations from the forward. If $z_{2.5\%}$ moves from about 1 in the morning to 5–10 near expiry, `option_skew_025` measures near-ATM skew early in the session and far-wing skew late in it. Its time-of-day profile is then mostly mechanical.

In [ ]:
features["z_of_k025"] = 0.025 / features["atm_sigma_sqrt_t"]
features["z_of_k050"] = 0.050 / features["atm_sigma_sqrt_t"]

z_table = (
    features.groupby(["underlying", "tte_bucket"], observed=True)["z_of_k025"]
    .median()
    .unstack("tte_bucket")
    .round(2)
)
print("Median z of the ±2.5% node, by asset and time to expiry:")
display(z_table)

beyond = (
    features.dropna(subset=["z_of_k025"])
    .groupby("underlying")["z_of_k025"]
    .agg(
        states="count",
        beyond_2sd_pct=lambda s: 100 * (s > 2).mean(),
        beyond_3sd_pct=lambda s: 100 * (s > 3).mean(),
        beyond_5sd_pct=lambda s: 100 * (s > 5).mean(),
    )
    .round(1)
)
print("Share of states where the ±2.5% node is more than 2, 3 or 5 expected moves away:")
display(beyond)

plot_median_by_tte(features, ["z_of_k025"], "Median z of the ±2.5% node through the session (log scale)", logy=True)

## 7. Mechanical drift of fixed-$k$ features

If fixed-$k$ features were stationary through the session, their median would be flat in time to expiry. The plots and the $\eta^2$ table measure how much of each feature's variance the clock bucket explains, per asset:

$$
\eta^2 = \frac{\sum_t (\bar x_{b(t)} - \bar x)^2}{\sum_t (x_t - \bar x)^2}
$$

where $\bar x_{b(t)}$ is the mean of the feature in the time-to-expiry bucket of state $t$. A high $\eta^2$ means the feature mostly encodes the clock. Part D compares this with the standardized-moneyness prototype.

In [ ]:
plot_median_by_tte(
    features,
    [c for c in ["option_iv_atm_forward", "option_skew_025", "option_curvature_025"] if c in features.columns],
    "Fixed-k features: median by time to expiry",
)
plot_median_by_tte(
    features,
    [c for c in ["rr_25d", "bf_25d"] if c in features.columns],
    "Delta features: median by time to expiry",
)

eta_rows = []
for asset, g in features.groupby("underlying"):
    for col in [c for c in OPTION_LEVEL if c in g.columns]:
        eta_rows.append({"underlying": asset, "feature": col, "eta2_tte": eta_squared(g, col)})
eta_fixed = pd.DataFrame(eta_rows).pivot(index="feature", columns="underlying", values="eta2_tte")
print("η² of the time-to-expiry bucket, per asset (fixed-k and delta features):")
display(eta_fixed.round(3))

# Part D — What the Surface Is Made Of (audit B2)

## 8. Node-level IV source

The preferred surface takes, per contract, the bid/ask IV midpoint when both sides exist, and otherwise Deribit's `mark_iv`. Each interpolated node is bracketed by two contracts. A node is classified as:

- **quote:** both bracketing contracts use the bid/ask midpoint;
- **mark:** both use Deribit's mark model;
- **mixed:** one of each.

A mark-sourced node reflects Deribit's own pricing model, not traded or quoted prices. The long-form query below joins `smile_node_state` to `normalized_option_chain` for the primary option family. It is the slowest step in the notebook.

In [ ]:
nodes = q(
    """
    WITH prim AS (
        SELECT DISTINCT option_family_id
        FROM primary_option_surface_feature_state
    ),
    chain AS MATERIALIZED (
        SELECT n.state_id, n.instrument_name, n.preferred_iv_source
        FROM normalized_option_chain n
        JOIN prim p ON n.option_family_id = p.option_family_id
        WHERE n.surface_base_eligible
    ),
    nodes AS (
        SELECT
            s.state_id, s.state_time, s.underlying, s.session_id, s.minutes_to_expiry,
            s.option_family_id, s.target_log_moneyness, s.interpolated_iv,
            s.interpolation_span_log_moneyness,
            s.left_instrument_name, s.right_instrument_name
        FROM smile_node_state s
        JOIN prim p ON s.option_family_id = p.option_family_id
        WHERE s.surface_source = 'preferred'
          AND s.interpolated_iv IS NOT NULL
    )
    SELECT
        nodes.* EXCLUDE (left_instrument_name, right_instrument_name),
        l.preferred_iv_source AS left_source,
        r.preferred_iv_source AS right_source
    FROM nodes
    LEFT JOIN chain l ON l.state_id = nodes.state_id AND l.instrument_name = nodes.left_instrument_name
    LEFT JOIN chain r ON r.state_id = nodes.state_id AND r.instrument_name = nodes.right_instrument_name
    """,
    "preferred smile nodes with source",
)
nodes["state_time"] = pd.to_datetime(nodes["state_time"], utc=True)
is_quote = lambda s: s.eq("bid_ask_iv_mid")
nodes["node_source"] = np.select(
    [is_quote(nodes["left_source"]) & is_quote(nodes["right_source"]),
     nodes["left_source"].eq("mark_iv") & nodes["right_source"].eq("mark_iv")],
    ["quote", "mark"],
    default="mixed",
)
nodes["tte_bucket"] = pd.cut(nodes["minutes_to_expiry"], bins=TTE_EDGES, labels=TTE_LABELS, right=True)

core = nodes[nodes["target_log_moneyness"].round(4).isin([-0.025, 0.0, 0.025])]
mark_share = (
    core.assign(any_mark=core["node_source"].ne("quote"))
    .groupby(["underlying", "tte_bucket"], observed=True)["any_mark"]
    .mean().mul(100).unstack("tte_bucket").round(1)
)
print("Core nodes (k = 0, ±2.5%) with at least one mark-sourced side, % — by asset and time to expiry:")
display(mark_share)

by_node = (
    nodes.assign(any_mark=nodes["node_source"].ne("quote"))
    .groupby(["underlying", "target_log_moneyness"])["any_mark"]
    .mean().mul(100).unstack("target_log_moneyness").round(1)
)
print("All nodes with at least one mark-sourced side, % — by asset and node:")
display(by_node)

print("Node source composition, all nodes:")
display(pd.crosstab(nodes["underlying"], nodes["node_source"], normalize="index").mul(100).round(1))

## 9. Do source switches create artificial changes?

Consecutive states (same session, exactly 5 minutes apart) are compared at the ATM node and at the ±2.5% nodes. If the node source changes between the two states (for example quote → mark because a bid disappeared), the IV change mixes a market move with a model switch.

If the dispersion of changes with a source switch is much larger than without one, the pipeline should compute changes only when both endpoints share a source, as audit B2 proposed.

In [ ]:
nd = nodes.sort_values(["underlying", "target_log_moneyness", "state_time"]).copy()
grp = nd.groupby(["underlying", "target_log_moneyness"], sort=False)
nd["prev_iv"] = grp["interpolated_iv"].shift(1)
nd["prev_source"] = grp["node_source"].shift(1)
nd["prev_time"] = grp["state_time"].shift(1)
nd["prev_session"] = grp["session_id"].shift(1)
ok = (
    (nd["state_time"] - nd["prev_time"] == pd.Timedelta(minutes=5))
    & (nd["session_id"] == nd["prev_session"])
)
nd = nd[ok].copy()
nd["d_iv"] = nd["interpolated_iv"] - nd["prev_iv"]
nd["transition"] = np.where(
    nd["node_source"] == nd["prev_source"], "same:" + nd["node_source"], "switch"
)

switch_summary = (
    nd[nd["target_log_moneyness"].round(4).isin([-0.025, 0.0, 0.025])]
    .groupby(["underlying", "transition"])["d_iv"]
    .agg(
        n="count",
        std="std",
        mad=lambda s: 1.4826 * (s - s.median()).abs().median(),
        p99_abs=lambda s: s.abs().quantile(0.99),
    )
    .round(4)
)
print("Core-node 5m IV changes by source transition:")
display(switch_summary)

switch_share = (
    nd.assign(sw=nd["transition"].eq("switch"))
    .groupby(["underlying", "tte_bucket"], observed=True)["sw"]
    .mean().mul(100).unstack("tte_bucket").round(1)
)
print("Share of node changes that cross a source switch, % — by time to expiry:")
display(switch_share)

# Part E — Standardized-Moneyness Prototype

## 10. Build a $z$-grid from the long-form chain

For each state of the primary family, every surface-eligible OTM contract (puts below the forward, calls above) is placed at

$$
z = \frac{\ln(K/F)}{\sigma_{ATM}\sqrt{\tau}}
$$

and IV is interpolated linearly in $k$ to the nodes $z \in \{-1, -0.5, 0, 0.5, 1\}$. As in the production grid, there is **no extrapolation**: a node exists only if strikes bracket it.

Two quality measures are kept per node:

- the interpolation span in $z$ units (near expiry, strike spacing becomes large relative to $\sigma\sqrt\tau$, so a $z$-node may sit between two distant strikes);
- whether the bracketing contracts are quote- or mark-sourced.

This is the honest trade-off of $z$-space: it fixes the meaning of each node, but near expiry the strike grid is coarse in $z$.

In [ ]:
Z_NODES = [-1.0, -0.5, 0.0, 0.5, 1.0]

chain = q(
    """
    WITH prim AS (
        SELECT DISTINCT option_family_id FROM primary_option_surface_feature_state
    )
    SELECT
        n.state_id,
        n.log_moneyness,
        n.preferred_iv,
        n.preferred_iv_source
    FROM normalized_option_chain n
    JOIN prim p ON n.option_family_id = p.option_family_id
    WHERE n.surface_base_eligible
      AND n.preferred_iv > 0
      AND n.log_moneyness IS NOT NULL
    """,
    "long-form primary chain",
)

scale = (
    features[["state_id", "atm_sigma_sqrt_t"]]
    .dropna()
    .query("atm_sigma_sqrt_t > 0")
    .set_index("state_id")["atm_sigma_sqrt_t"]
)
chain = chain[chain["state_id"].isin(scale.index)]
chain = chain.sort_values(["state_id", "log_moneyness"]).reset_index(drop=True)
state_codes, state_index = pd.factorize(chain["state_id"], sort=False)
OFFSET = 10.0  # |log-moneyness| < 5 for any listed strike
key = state_codes * OFFSET + chain["log_moneyness"].to_numpy()
assert np.all(np.diff(key) >= 0), "chain must be sorted by state, then k"

iv = chain["preferred_iv"].to_numpy()
src_quote = chain["preferred_iv_source"].eq("bid_ask_iv_mid").to_numpy()
k_arr = chain["log_moneyness"].to_numpy()
n = len(chain)
s_state = scale.reindex(state_index).to_numpy()
codes_target = np.arange(len(state_index))

z_out = pd.DataFrame({"state_id": state_index})
for z in Z_NODES:
    k_target = z * s_state
    tkey = codes_target * OFFSET + k_target
    pos = np.searchsorted(key, tkey, side="left")
    pos_c = np.clip(pos, 0, n - 1)
    # Exact hit: compare on the state code and raw k, not on the packed key
    # (the packed key loses precision at large state codes).
    exact = (pos < n) & (state_codes[pos_c] == codes_target) & (np.abs(k_arr[pos_c] - k_target) <= 1e-12)
    right = np.where(exact, pos_c, pos)
    left = np.where(exact, pos_c, pos - 1)
    valid = (left >= 0) & (right < n)
    left_c, right_c = np.clip(left, 0, n - 1), np.clip(right, 0, n - 1)
    valid &= (state_codes[left_c] == codes_target) & (state_codes[right_c] == codes_target)
    span_k = k_arr[right_c] - k_arr[left_c]
    w = np.where(span_k > 0, (k_target - k_arr[left_c]) / np.where(span_k > 0, span_k, 1.0), 0.0)
    iv_z = iv[left_c] + w * (iv[right_c] - iv[left_c])
    tag = "00" if z == 0 else f"{z:+.1f}".replace("+", "p").replace("-", "m").replace(".", "")
    z_out[f"iv_z_{tag}"] = np.where(valid, iv_z, np.nan)
    z_out[f"span_z_{tag}"] = np.where(valid, span_k / s_state, np.nan)
    z_out[f"quote_z_{tag}"] = np.where(valid, src_quote[left_c] & src_quote[right_c], np.nan)

z_out["z_skew_05"] = z_out["iv_z_p05"] - z_out["iv_z_m05"]
z_out["z_skew_10"] = z_out["iv_z_p10"] - z_out["iv_z_m10"]
z_out["z_curv_05"] = (z_out["iv_z_p05"] + z_out["iv_z_m05"]) / 2 - z_out["iv_z_00"]
z_out["z_curv_10"] = (z_out["iv_z_p10"] + z_out["iv_z_m10"]) / 2 - z_out["iv_z_00"]
z_out["z_max_span_05"] = z_out[["span_z_m05", "span_z_00", "span_z_p05"]].max(axis=1)
z_out["z_max_span_10"] = z_out[["span_z_m10", "span_z_00", "span_z_p10"]].max(axis=1)

zf = features.merge(z_out, on="state_id", how="left")

# Consistency check: the z = 0 node must equal the pipeline's ATM IV (both are k = 0).
chk = zf[["iv_z_00", "option_iv_atm_forward"]].dropna()
diff = (chk.iloc[:, 0] - chk.iloc[:, 1]).abs()
print(f"z=0 vs pipeline ATM IV: n={len(chk):,} | identical (<1e-6): {100 * (diff < 1e-6).mean():.2f}% | max |diff| = {diff.max():.3e}")
print("Any mismatch must come from tie-breaking between a put and a call at the same strike; inspect if large.")
print(f"States with a z-grid: {z_out['iv_z_00'].notna().sum():,} of {len(z_out):,}")

## 11. Coverage and interpolation quality of the $z$-grid

A $z$-node bracketed by strikes several $\sigma\sqrt\tau$ apart is mostly an interpolation artifact. The table reports node availability and the median span in $z$ units by time to expiry. A practical acceptance rule would be a maximum span in $z$ (for example 1.0) — the table shows what such a rule would cost.

In [ ]:
z_cov = (
    zf.groupby(["underlying", "tte_bucket"], observed=True)
    .agg(
        states=("state_id", "count"),
        z05_available_pct=("z_skew_05", lambda s: 100 * s.notna().mean()),
        z10_available_pct=("z_skew_10", lambda s: 100 * s.notna().mean()),
        median_span_z05=("z_max_span_05", "median"),
        span_z05_le_1_pct=("z_max_span_05", lambda s: 100 * (s <= 1.0).mean()),
        quote_atm_pct=("quote_z_00", lambda s: 100 * s.mean()),
    )
    .round(2)
)
display(z_cov)

kcov = (
    features.assign(core=as_bool(features["core_smile_usable"]) if "core_smile_usable" in features.columns else False)
    .groupby(["underlying", "tte_bucket"], observed=True)["core"].mean().mul(100).round(1)
    .unstack("tte_bucket")
)
print("For comparison — fixed-k core usable (%):")
display(kcov)

## 12. Is the $z$-grid more stationary through the session?

The same $\eta^2$ diagnostic as Section 7, now comparing fixed-$k$ and $z$ features on the states where both exist. Skew and curvature are also shown **relative to ATM IV**, because ATM IV itself has an intraday profile.

Reading the result:

- if $z$-features have clearly lower $\eta^2$ than their fixed-$k$ counterparts, the fixed-$k$ grid is mostly measuring the clock and audit A2 holds;
- if they are similar, the fixed-$k$ grid is not the main source of TTE dependence, and the remaining profile is genuine intraday seasonality (to be removed by the clock baseline in the framework's Step 2).

In [ ]:
zf["skew_025_rel"] = zf["option_skew_025"] / zf["option_iv_atm_forward"]
zf["curv_025_rel"] = zf["option_curvature_025"] / zf["option_iv_atm_forward"]
zf["z_skew_05_rel"] = zf["z_skew_05"] / zf["option_iv_atm_forward"]
zf["z_curv_05_rel"] = zf["z_curv_05"] / zf["option_iv_atm_forward"]

PAIRS = [
    ("skew", "option_skew_025", "z_skew_05"),
    ("curvature", "option_curvature_025", "z_curv_05"),
    ("skew / ATM", "skew_025_rel", "z_skew_05_rel"),
    ("curvature / ATM", "curv_025_rel", "z_curv_05_rel"),
]

rows = []
for asset, g in zf.groupby("underlying"):
    for label, kcol, zcol in PAIRS:
        both = g.dropna(subset=[kcol, zcol])
        rows.append({
            "underlying": asset, "measure": label, "n_both": len(both),
            "eta2_fixed_k": eta_squared(both, kcol),
            "eta2_z": eta_squared(both, zcol),
        })
stationarity = pd.DataFrame(rows)
stationarity["z_minus_k"] = stationarity["eta2_z"] - stationarity["eta2_fixed_k"]
display(stationarity.round(3))

plot_median_by_tte(zf, ["option_skew_025", "z_skew_05"], "Skew: fixed k = ±2.5% vs z = ±0.5")
plot_median_by_tte(zf, ["option_curvature_025", "z_curv_05"], "Curvature: fixed k = ±2.5% vs z = ±0.5")

# Part F — Dynamics

## 13. Surface-change distributions and the settlement window (audit B4)

Deribit daily options settle to the 30-minute TWAP of the index from 07:30 to 08:00 UTC. During the last six states (`minutes_to_expiry ≤ 30`) part of the settlement price is already fixed, so quotes and implied volatilities stop behaving like those of a European option. The comparison below shows whether level and change distributions break inside that window.

In [ ]:
dyn_summary = feature_summary(features, OPTION_DYNAMIC, group_col="underlying")
display(dyn_summary)

settle_rows = []
for asset, g in features.groupby("underlying"):
    for col in ["option_iv_atm_forward", "atm_iv_change_5m", "skew_025_change_5m"]:
        if col not in g.columns:
            continue
        for label, part in [("settlement window", g[g["in_settlement_window"]]),
                            ("1h before window", g[g["minutes_to_expiry"].between(35, 90)]),
                            ("rest of session", g[g["minutes_to_expiry"] > 90])]:
            s = clean_numeric(part, col)
            settle_rows.append({
                "underlying": asset, "feature": col, "segment": label, "n": len(s),
                "median": s.median() if len(s) else np.nan,
                "median_abs": s.abs().median() if len(s) else np.nan,
                "p99_abs": s.abs().quantile(0.99) if len(s) else np.nan,
            })
settlement = pd.DataFrame(settle_rows)
display(settlement.pivot_table(index=["feature", "underlying"], columns="segment", values=["median_abs", "p99_abs"]).round(3))

last = features[features["minutes_to_expiry"] <= 120]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for asset, g in last.groupby("underlying"):
    m = g.groupby("minutes_to_expiry")["option_iv_atm_forward"].median().sort_index(ascending=False)
    axes[0].plot(m.index, m.values, label=asset)
    if "atm_iv_change_5m" in g.columns:
        a = g.groupby("minutes_to_expiry")["atm_iv_change_5m"].apply(lambda s: s.abs().median()).sort_index(ascending=False)
        axes[1].plot(a.index, a.values, label=asset)
for ax, title in zip(axes, ["median ATM IV", "median |ΔATM IV 5m|"]):
    ax.axvspan(0, 30, alpha=0.12)
    ax.invert_xaxis()
    ax.set_xlabel("minutes to expiry (shaded: settlement window)")
    ax.set_title(title)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 14. Persistence within the session

Level features that are persistent are candidate **structural** inputs; changes with near-zero autocorrelation behave like innovations. All autocorrelations are computed on the exact 5-minute grid and never cross the 08:00 boundary.

In [ ]:
acf_frames = []
for col in [c for c in ["option_iv_atm_forward", "option_skew_025", "option_curvature_025", "rr_25d",
                        "atm_iv_change_5m", "skew_025_change_5m"] if c in features.columns]:
    acf_frames.append(within_session_acf(features, col, max_lag=24))
for col in ["z_skew_05", "z_curv_05"]:
    acf_frames.append(within_session_acf(zf, col, max_lag=24))
option_acf = pd.concat(acf_frames, ignore_index=True)

summary_lags = [5, 30, 60, 120]
display(
    option_acf[option_acf["lag_minutes"].isin(summary_lags)]
    .pivot_table(index=["feature", "underlying"], columns="lag_minutes", values="acf")
    .round(3)
)

feats = option_acf["feature"].unique()
fig, axes = plt.subplots(math.ceil(len(feats) / 2), 2, figsize=(14, 3.6 * math.ceil(len(feats) / 2)))
for ax, feat in zip(np.atleast_1d(axes).ravel(), feats):
    for asset, g in option_acf[option_acf["feature"] == feat].groupby("underlying"):
        ax.plot(g["lag_minutes"], g["acf"], linewidth=1, label=asset)
    ax.axhline(0, linewidth=0.8)
    ax.set_title(feat, fontsize=10)
np.atleast_1d(axes).ravel()[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

# Part G — Robustness

## 15. Linear USDC vs inverse families (BTC, ETH)

BTC and ETH list both linear USDC and inverse options for the same expiry. The primary family is linear USDC. If both families give the same ATM IV and the same changes, the primary-family choice is harmless; a persistent gap or weak change correlation would mean one family's surface is dominated by stale or mark-sourced quotes.

In [ ]:
fam_cols = relation_columns("option_surface_feature_state")
if {"iv_atm_forward", "option_family_id", "state_id"}.issubset(fam_cols):
    fam = q(
        """
        SELECT state_id, state_time, underlying, option_family_id, instrument_type,
               iv_atm_forward, atm_iv_change_5m
        FROM option_surface_feature_state
        WHERE underlying IN ('BTC', 'ETH')
        """,
        "BTC/ETH families",
    )
    fam["family"] = np.where(fam["instrument_type"].str.lower().eq("linear"), "linear", "inverse")
    wide = fam.pivot_table(index=["underlying", "state_id"], columns="family",
                           values=["iv_atm_forward", "atm_iv_change_5m"], aggfunc="first")
    if {("iv_atm_forward", "linear"), ("iv_atm_forward", "inverse")}.issubset(wide.columns):
        wide["atm_gap"] = wide[("iv_atm_forward", "linear")] - wide[("iv_atm_forward", "inverse")]
        out = []
        for asset, g in wide.groupby(level="underlying"):
            ch = g[[("atm_iv_change_5m", "linear"), ("atm_iv_change_5m", "inverse")]].dropna()
            out.append({
                "underlying": asset,
                "states_both": int(g["atm_gap"].notna().sum()),
                "median_gap": g["atm_gap"].median(),
                "mad_gap": 1.4826 * (g["atm_gap"] - g["atm_gap"].median()).abs().median(),
                "p99_abs_gap": g["atm_gap"].abs().quantile(0.99),
                "corr_5m_changes": ch.corr().iloc[0, 1] if len(ch) > 20 else np.nan,
            })
        family_comparison = pd.DataFrame(out)
        display(family_comparison.round(4))
    else:
        family_comparison = pd.DataFrame()
        print("Only one family present for BTC/ETH; comparison skipped.")
else:
    family_comparison = pd.DataFrame()
    print("option_surface_feature_state lacks the needed columns; comparison skipped.")

## 16. Level drift across sessions

Median ATM IV and skew per session. Large session-to-session level shifts are expected in crypto (volatility regimes); they matter here because any later normalization must be fitted on training history only.

In [ ]:
session_levels = (
    features.groupby(["expiry_date", "underlying"])[
        [c for c in ["option_iv_atm_forward", "option_skew_025", "rr_25d"] if c in features.columns]
    ]
    .median()
    .unstack("underlying")
)
display(session_levels.round(3))

fig, ax = plt.subplots(figsize=(13, 4.5))
for asset in ASSETS:
    s = features[features["underlying"] == asset].groupby("expiry_date")["option_iv_atm_forward"].median()
    ax.plot(s.index, s.values, marker="o", ms=3, label=asset)
ax.set_title("Median ATM IV per session")
ax.legend(fontsize=8, ncol=4)
plt.tight_layout()
plt.show()

# Part H — Research Handoff

## 17. Interpretation checklist

Answer these before Notebook 04:

1. **B1:** how many valid option states does the futures gate remove, and on which dates? Is decoupling surface eligibility from the futures stream justified?
2. **A2:** how far does the ±2.5% node travel in $z$ through the session, and for which assets is it beyond 3σ for most of the day?
3. **A2:** do $z$-space skew and curvature have materially lower $\eta^2$ than the fixed-$k$ versions? For which assets?
4. **A2 trade-off:** where does the $z$-grid lose coverage or need large spans (near expiry, coarse strike grids)?
5. **B2:** what share of core nodes is mark-sourced, by asset and time to expiry? Above what share should an asset's option features be treated as Deribit-model information rather than market information?
6. **B2:** are 5-minute changes across a source switch materially noisier than same-source changes?
7. **B4:** do levels or changes break inside the settlement window?
8. Which option features are persistent enough to be structural, and which behave as innovations?
9. Do the BTC/ETH linear and inverse families agree?
10. Is any of this based on too little history (16 sessions) to support a production decision?

**Do not change the production surface here.** A decision on $z$-space (A2), source-consistent changes (B2), futures decoupling (B1) and a settlement flag (B4) is a `src/` change to be proposed with this notebook as evidence.

In [ ]:
surface_distribution_report = {
    "futures_gate_loss": gate,
    "availability_by_tte": availability.reset_index(),
    "level_by_asset": level_by_asset,
    "z_of_k025_by_tte": z_table.reset_index(),
    "eta2_fixed": eta_fixed.reset_index(),
    "core_mark_share_by_tte": mark_share.reset_index(),
    "source_switch_changes": switch_summary.reset_index(),
    "z_grid_coverage": z_cov.reset_index(),
    "stationarity_k_vs_z": stationarity,
    "settlement_window": settlement,
    "option_acf": option_acf,
    "family_comparison": family_comparison,
}
print("Surface-distribution report sections:")
for name, frame in surface_distribution_report.items():
    print(f"  {name:<26} {len(frame):>8,} rows")

## 18. Close the read-only connection

In [ ]:
con.close()
print("DuckDB connection closed.")